In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Снимаем лимит на размер панорам
Image.MAX_IMAGE_PIXELS = None

# Настраиваем устройство (GPU, если доступно, иначе CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Будем обучать на: {device}")

# Вспомним наш путь к данным
DATA_DIR = Path("Задача 3. Скажи мне, кто твой шлиф")

c:\Users\user\Desktop\hackathon\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Будем обучать на: cpu


In [2]:
def generate_masks(img_rgb):
    """
    Генерирует маски классов на основе визуальных признаков из ТЗ.
    Возвращает тензор масок формы (H, W), где каждый пиксель - это номер класса.
    Классы: 0 - Фон, 1 - Обычные сульфиды, 2 - Тонкие сульфиды, 3 - Тальк
    """
    h, w, _ = img_rgb.shape
    mask = np.zeros((h, w), dtype=np.uint8) # 0 = Фон
    
    # 1. Извлечение маски талька (Поиск цветной линии)
    # Предположим, разметка сделана ярко-зеленой линией. ТУТ НУЖНО ПОДСТАВИТЬ СВОЙ ЦВЕТ!
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    lower_talc = np.array([40, 100, 100]) # Настрой эти пороги HSV
    upper_talc = np.array([80, 255, 255])
    talc_mask = cv2.inRange(hsv, lower_talc, upper_talc)
    mask[talc_mask > 0] = 3 # Класс 3 - Тальк
    
    # 2. Выделение сульфидов (Светлые области)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, sulfide_mask = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY) # Порог яркости > 200
    
    # ПРИМЕЧАНИЕ ДЛЯ ХАКАТОНА: 
    # В идеале здесь нужно добавить логику разделения сульфидов на обычные (класс 1) 
    # и тонкие (класс 2), например, через анализ площади контуров (крупные - класс 1, мелкие - класс 2).
    # Пока для бейзлайна запишем все сульфиды как класс 1, если там нет талька.
    sulfide_coords = (sulfide_mask > 0) & (mask == 0)
    mask[sulfide_coords] = 1 
    
    return mask

# Функция безопасного чтения (из прошлого ноутбука)
def read_image_safe(path):
    file_bytes = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img is not None:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return None

In [8]:
class OreDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = read_image_safe(img_path)
        
        if image is None:
            # Защита от битых файлов: если файл не прочитался, берем другой случайный
            return self.__getitem__(np.random.randint(len(self.file_paths)))
            
        # Генерируем маску
        mask = generate_masks(image)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        # Конвертируем маску в LongTensor для CrossEntropyLoss
        return image, mask.long()

# Аугментации: Вырезаем случайный кусок 512x512 и нормализуем под ImageNet
# Обновленные аугментации с защитой от маленьких картинок
train_transforms = A.Compose([
    # Если картинка меньше 512 по любой из сторон, дополняем её до 512 черными пикселями (0)
    A.PadIfNeeded(
        min_height=512, 
        min_width=512, 
        border_mode=cv2.BORDER_CONSTANT, 
        value=0, 
        mask_value=0, 
        always_apply=True
    ),
    # Теперь RandomCrop гарантированно отработает, так как размеры точно >= 512
    A.RandomCrop(height=512, width=512, always_apply=True),
    
    # Стандартные аугментации для материаловедения
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Собираем пути к картинкам (возьмем, например, все файлы из "ч2" для обучения)
train_files = list((DATA_DIR / "Фото руд по сортам. ч2").rglob("*.jpg")) + \
              list((DATA_DIR / "Фото руд по сортам. ч2").rglob("*.png"))

print(f"Найдено файлов для обучения: {len(train_files)}")

train_dataset = OreDataset(train_files, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0) # num_workers=0 для Windows

Найдено файлов для обучения: 1001


C:\Users\user\AppData\Local\Temp\ipykernel_5568\3541819893.py:32: UserWarning: Argument(s) 'value, mask_value, always_apply' are not valid for transform PadIfNeeded
  A.PadIfNeeded(
C:\Users\user\AppData\Local\Temp\ipykernel_5568\3541819893.py:41: UserWarning: Argument(s) 'always_apply' are not valid for transform RandomCrop
  A.RandomCrop(height=512, width=512, always_apply=True),


In [9]:
# Создаем модель U-Net
# У нас 4 класса: 0-Фон, 1-Обычные, 2-Тонкие, 3-Тальк
NUM_CLASSES = 4

model = smp.Unet(
    encoder_name="resnet34",        # Легкий и быстрый энкодер
    encoder_weights="imagenet",     # Предобученные веса
    in_channels=3,                  # RGB изображения
    classes=NUM_CLASSES,            # Количество классов на выходе
)

model = model.to(device)

# Функция потерь: CrossEntropy хорошо работает для мультиклассовой сегментации
criterion = nn.CrossEntropyLoss()

# Оптимизатор
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Модель успешно инициализирована!")

Модель успешно инициализирована!


In [ ]:
# Вместо from tqdm.notebook import tqdm пишем:
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

EPOCHS = 3 # Для начала хватит 3 эпох, чтобы проверить пайплайн
scaler = GradScaler()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    # Оборачиваем dataloader в tqdm для красивого прогресс-бара
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}")
    
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Training
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Эпоха {epoch+1} завершена. Средний лосс: {epoch_loss:.4f}")

# Сохраняем веса бейзлайн-модели
torch.save(model.state_dict(), "baseline_unet.pth")
print("Веса модели сохранены в baseline_unet.pth!")

C:\Users\user\AppData\Local\Temp\ipykernel_5568\1065742158.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\user\AppData\Local\Temp\ipykernel_5568\1065742158.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
